In [33]:
from abc import ABC, abstractmethod
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go  

In [ ]:
# ==========================================
# 1. CLASSE GENITORE (BASE)
# ==========================================
class BaseCryptoStrategy(ABC):
    """
    Classe genitore che gestisce il recupero dati, le statistiche e i grafici interattivi.
    Le classi figlie devono implementare solo la logica di 'run_strategy'.
    """
    
    def __init__(self, initial_value: float = 10000.0):
        self.initial_value = initial_value
        self.data_dict = {}
        self.assets = []
        self.index_df = None
        self.benchmark_df = None  
        self.benchmark_ticker = 'BTC-USD' 
        self.rebalance_dates = [] # <-- Nuovo attributo per tracciare le date di ribilanciamento

    def import_data(self, assets: list, timeframe: str = "1y"):
        """Scarica e prepara i dati base (Prezzo, Volume, Circulating Supply) per la lista di asset."""
        print(f"Inizio download dati per il periodo '{timeframe}'...")
        self.assets = []
        
        for ticker_symbol in assets:
            try:
                ticker = yf.Ticker(ticker_symbol)
                df_raw = ticker.history(period=timeframe)
                
                if df_raw.empty:
                    print(f"Nessun dato per {ticker_symbol}. Salto.")
                    continue
                    
                circulating_supply = ticker.info.get('circulatingSupply')
                if not circulating_supply:
                    print(f"Circulating Supply mancante per {ticker_symbol}. Salto.")
                    continue
                    
                df_final = df_raw[['Close', 'Volume']].copy()
                df_final['Circulating Supply'] = circulating_supply
                df_final.columns = ['Prezzo di Chiusura', 'Volume 24h', 'Circulating Supply']
                
                self.data_dict[ticker_symbol] = df_final
                self.assets.append(ticker_symbol)
                
            except Exception as e:
                print(f"Errore durante il recupero di {ticker_symbol}: {e}")

        # --- GESTIONE DEL BENCHMARK ---
        if self.benchmark_ticker in self.data_dict:
            self.benchmark_df = self.data_dict[self.benchmark_ticker]
        else:
            print(f"\n{self.benchmark_ticker} non presente nel dizionario. Avvio download in un df separato...")
            try:
                ticker_bench = yf.Ticker(self.benchmark_ticker)
                df_bench_raw = ticker_bench.history(period=timeframe)
                
                if not df_bench_raw.empty:
                    df_bench_final = df_bench_raw[['Close']].copy()
                    df_bench_final.columns = ['Prezzo di Chiusura']
                    self.benchmark_df = df_bench_final
                    print(f"Benchmark {self.benchmark_ticker} completato con successo.\n")
                else:
                    print(f"Nessun dato trovato per il benchmark {self.benchmark_ticker}.\n")
            except Exception as e:
                print(f"Errore durante il download del benchmark {self.benchmark_ticker}: {e}\n")

    @abstractmethod
    def run_strategy(self, **kwargs):
        """Metodo astratto: ogni classe figlia deve definire la propria logica di backtest."""
        pass

    def plot_results(self):
        """Grafica l'andamento della strategia confrontato con il benchmark (Interattivo)."""
        if self.index_df is None:
            raise ValueError("Strategia non calcolata. Esegui run_strategy() prima di plottare.")
            
        fig = go.Figure()

        # --- PLOT STRATEGIA ---
        fig.add_trace(go.Scatter(
            x=self.index_df.index, 
            y=self.index_df['Valore Indice'],
            mode='lines',
            name='Indice Strategia',
            line=dict(color='#1f77b4', width=2)
        ))
        
        # --- PLOT BENCHMARK ---
        if self.benchmark_df is not None:
            common_idx = self.index_df.index.intersection(self.benchmark_df.index)
            if not common_idx.empty:
                btc_prices = self.benchmark_df['Prezzo di Chiusura'].loc[common_idx]
                btc_normalized = (btc_prices / btc_prices.iloc[0]) * self.initial_value
                
                fig.add_trace(go.Scatter(
                    x=btc_normalized.index, 
                    y=btc_normalized,
                    mode='lines',
                    name=f'{self.benchmark_ticker} (Benchmark)',
                    line=dict(color='#cc8306', width=2)
                ))
        
        # --- LINEA CAPITALE INIZIALE ---
        fig.add_hline(
            y=self.initial_value, 
            line_dash="dash", 
            line_color="red", 
            annotation_text=f"Capitale Iniziale ({self.initial_value}$)", 
            annotation_position="bottom right"
        )

        # --- LINEE DEI RIBILANCIAMENTI ---
        # Aggiungiamo una linea verticale per ogni data in cui è avvenuto un ribilanciamento
        if self.rebalance_dates:
            for date in self.rebalance_dates:
                fig.add_vline(
                    x=date, 
                    line_width=1, 
                    line_dash="dot", 
                    line_color="green",
                    opacity=0.6
                )
            # Aggiungo una riga invisibile solo per far comparire la legenda dei ribilanciamenti
            fig.add_trace(go.Scatter(
                x=[None], y=[None], mode='lines',
                line=dict(color='green', dash='dot'),
                name='Ribilanciamento'
            ))

        # --- LAYOUT INTERATTIVO ---
        fig.update_layout(
            title="Performance Strategia vs BTC-USD (Scala Logaritmica)",
            xaxis_title="Data",
            yaxis_title="Valore del Portafoglio ($)",
            hovermode="x unified", # Mostra i dati di tutte le curve passando il mouse su un punto
            template="plotly_white",
            legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
            yaxis_type="log"  # Scala logaritmica
        )
        
        fig.show()

    def calculate_stats(self):
        """Calcola e restituisce le statistiche della strategia rispetto al benchmark."""
        if self.index_df is None:
            raise ValueError("Strategia non calcolata. Esegui run_strategy().")
        
        strat_returns = self.index_df['Valore Indice'].pct_change().dropna()
        stats = {'Strategia': self._compute_metrics(strat_returns)}
        
        if self.benchmark_df is not None:
            common_idx = self.index_df.index.intersection(self.benchmark_df.index)
            if not common_idx.empty:
                btc_prices = self.benchmark_df['Prezzo di Chiusura'].loc[common_idx]
                btc_returns = btc_prices.pct_change().dropna()
                stats[self.benchmark_ticker] = self._compute_metrics(btc_returns)
            
        return pd.DataFrame(stats).round(4)

    def _compute_metrics(self, returns: pd.Series) -> dict:
        """Helper interno per il calcolo delle metriche finanziarie su 365 giorni."""
        trading_days = 365
        ann_return = returns.mean() * trading_days
        ann_volatility = returns.std() * np.sqrt(trading_days)
        sharpe_ratio = ann_return / ann_volatility if ann_volatility != 0 else 0
        
        cumulative_returns = (1 + returns).cumprod()
        peak = cumulative_returns.cummax()
        max_drawdown = ((cumulative_returns - peak) / peak).min()
        
        return {
            'Ritorno Annualizzato (%)': ann_return * 100,
            'Volatilità Annualizzata (%)': ann_volatility * 100,
            'Sharpe Ratio': sharpe_ratio,
            'Max Drawdown (%)': max_drawdown * 100
        }

In [35]:
# ==========================================
# 2. CLASSE FIGLIA (MARKET CAP INDEX STRATEGY)
# ==========================================
class MarketCapIndexStrategy(BaseCryptoStrategy):
    """
    Strategia che crea un indice dinamico ponderato per market cap.
    Le crypto vengono aggiunte all'indice quando raggiungono il minimo market cap (default 100M USD).
    Vengono rimosse quando scendono sotto la soglia.
    Il portafoglio è sempre ponderato per market cap tra i crypto elegibili.
    """
    
    def __init__(self, initial_value: float = 10000.0):
        super().__init__(initial_value)
        self.composition_history = {}  # Traccia la composizione nel tempo
        self.composition_dates = []     # Date di ribilanciamento
    
    def run_strategy(self, rebalance_freq='ME', min_marketcap=100_000_000, **kwargs):
        """
        rebalance_freq: Frequenza di ribilanciamento. 'ME' = fine mese, 'QE' = fine trimestre, 'YE' = fine anno.
        min_marketcap: Market cap minimo (in USD) per essere incluso nell'indice. Default 100M.
        """
        if not self.data_dict:
            raise ValueError("Nessun dato disponibile. Esegui import_data() per primo.")
            
        print(f"\nAvvio backtest: Market Cap Index con soglia minima ${min_marketcap/1e6:.0f}M (frequenza ribilanciamento: {rebalance_freq})...")
        
        df_prices = pd.DataFrame()
        df_mktcap = pd.DataFrame()
        
        # 1. Allineamento dati e calcolo della Market Cap
        for ticker in self.assets:
            df = self.data_dict[ticker].copy()
            df['MarketCap'] = df['Prezzo di Chiusura'] * df['Circulating Supply']
            
            df.index = pd.to_datetime(df.index)
            df_prices[ticker] = df['Prezzo di Chiusura']
            df_mktcap[ticker] = df['MarketCap']
            
        df_prices.ffill(inplace=True)
        df_mktcap.ffill(inplace=True)
        df_prices.dropna(how='all', inplace=True)
        
        dates = df_prices.index
        first_date = dates[0]
        
        # 2. Identificazione dei giorni di ribilanciamento
        rebalance_dates = df_prices.resample(rebalance_freq).last().index
        rebalance_dates = [d for d in rebalance_dates if d in dates]
        
        portfolio_value = self.initial_value
        shares = {}
        current_index_cryptos = []
        index_values = []
        prev_index_cryptos = []
        
        # 3. Loop giornaliero: simulazione day-by-day
        for current_date in dates:
            
            # --- A. AGGIORNAMENTO VALORE PORTAFOGLIO ---
            if shares:
                portfolio_value = sum(shares[ticker] * df_prices.loc[current_date, ticker] for ticker in current_index_cryptos if ticker in shares)
            
            index_values.append(portfolio_value)
            
            # --- B. CONTROLLO RIBILANCIAMENTO ---
            if current_date == first_date or current_date in rebalance_dates:
                
                # Prende la Mkt Cap di TUTTE le crypto in questa specifica data
                daily_mktcap = df_mktcap.loc[current_date].dropna()
                
                # Filtra i crypto che raggiungono il minimo market cap
                eligible_cryptos = daily_mktcap[daily_mktcap >= min_marketcap]
                current_index_cryptos = eligible_cryptos.index.tolist()
                
                total_mktcap = eligible_cryptos.sum()
                if total_mktcap == 0:
                    continue
                
                # Calcola i pesi basati su market cap
                weights = (eligible_cryptos / total_mktcap).to_dict()
                
                # Traccia composizione
                self.composition_history[current_date] = weights.copy()
                self.composition_dates.append(current_date)
                
                # Stampa quando nuovi crypto entrano/escono
                new_entries = set(current_index_cryptos) - set(prev_index_cryptos)
                removed = set(prev_index_cryptos) - set(current_index_cryptos)
                
                if new_entries and current_date != first_date:
                    print(f"  {current_date.strftime('%Y-%m-%d')}: +{', '.join(sorted(new_entries))}")
                if removed:
                    print(f"  {current_date.strftime('%Y-%m-%d')}: -{', '.join(sorted(removed))}")
                
                prev_index_cryptos = current_index_cryptos.copy()
                
                # Ricalcola le quote per tutti i crypto nell'indice
                shares = {}
                for ticker in current_index_cryptos:
                    allocated_capital = portfolio_value * weights[ticker]
                    price = df_prices.loc[current_date, ticker]
                    shares[ticker] = allocated_capital / price if price > 0 else 0
                    
        # 4. Salvataggio della serie storica
        self.index_df = pd.DataFrame({'Date': dates, 'Valore Indice': index_values})
        self.index_df.set_index('Date', inplace=True)
        
        print(f"\nBacktest completato. Composizione FINALE dell'indice al {dates[-1].strftime('%Y-%m-%d')}:")
        final_mktcap = df_mktcap.loc[dates[-1]]
        for ticker in current_index_cryptos:
            if ticker in weights:
                print(f"- {ticker}: {weights[ticker]:.2%} (Mkt Cap: ${final_mktcap[ticker]:,.0f})")
    
    def plot_composition(self):
        """Visualizza come cambia la composizione dell'indice nel tempo."""
        if not self.composition_history:
            raise ValueError("Composizione non tracciata. Esegui run_strategy() prima.")
        
        # Crea un DataFrame con la composizione nel tempo
        composition_df = pd.DataFrame(self.composition_history).T
        composition_df.index = pd.to_datetime(composition_df.index)
        composition_df = composition_df.fillna(0)
        
        # Ordina i ticker per frequenza di apparizione (i più frequenti in alto)
        ticker_freq = (composition_df > 0).sum().sort_values(ascending=False)
        composition_df = composition_df[ticker_freq.index]
        
        # Crea un grafico stacked area
        fig = go.Figure()
        
        for ticker in composition_df.columns:
            fig.add_trace(go.Scatter(
                x=composition_df.index,
                y=composition_df[ticker],
                mode='lines',
                name=ticker,
                stackgroup='one',
                fillcolor=None,
                hovertemplate='<b>%{fullData.name}</b><br>%{x|%Y-%m-%d}<br>% del portafoglio: %{y:.1%}<extra></extra>'
            ))
        
        fig.update_layout(
            title="Evoluzione della Composizione dell'Indice nel Tempo",
            xaxis_title="Data",
            yaxis_title="Peso nel Portafoglio (%)",
            hovermode="x unified",
            template="plotly_white",
            yaxis=dict(tickformat=".0%"),
            height=600,
            legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
        )
        
        fig.show()


In [ ]:
if __name__ == "__main__":
    # 1. Define the list of assets to include in the portfolio
    # It is recommended to include "BTC-USD" so it is used as benchmark in charts and statistics

    Conio_coin = [
        "BTC-USD", "ETH-USD", "XRP-USD", "SOL-USD", "DOGE-USD", 
        "ADA-USD", "LINK-USD", "AVAX-USD", "DOT-USD", 
        "UNI7083-USD", "SKY33038-USD", "NEAR-USD", "ATOM-USD", 
        "POL28321-USD", "ALGO-USD", "APT21794-USD", "ARB11841-USD", "STX4847-USD", 
        "INJ-USD", "TIA-USD", "GRT6719-USD", "OP-USD", "SUI20947-USD", 
        "XLM-USD", "XPL-USD", "ONDO-USD", "HBAR-USD", 
        "FIL-USD", "AAVE-USD", "NIGHT39064-USD", "ETC-USD", "LTC-USD"
    ]
    
    # 2. Initialize the strategy with starting capital
    strategy = MarketCapIndexStrategy(initial_value=10000.0)
    
    # 3. Import data (e.g., full historical data)
    strategy.import_data(assets=Conio_coin, timeframe="max")
    
    # 4. Run the backtest with dynamic market cap index
    # Wrap in try-except in case of connection or missing data errors
    try:
        strategy.run_strategy(rebalance_freq='ME', min_marketcap=1_000_000_000)  # 1B USD minimum
        
        # 5. Calculate and print statistics
        print("\n--- Performance Statistics ---")
        stats_df = strategy.calculate_stats()
        print(stats_df)
        
        # 6. Display the chart of index performance
        strategy.plot_results()
        
        # 7. Display the composition evolution
        print("\n--- Index Composition Over Time ---")
        strategy.plot_composition()
        
    except ValueError as e:
        print(f"\nError during strategy execution: {e}")

Inizio download dati per il periodo 'max'...

Avvio backtest: Market Cap Index con soglia minima $1000M (frequenza ribilanciamento: ME)...
  2017-04-30: +LTC-USD
  2017-11-30: +ADA-USD, ETC-USD, ETH-USD, USDT-USD, XLM-USD, XRP-USD
  2017-12-31: +DOGE-USD, FIL-USD
  2018-03-31: -DOGE-USD
  2018-11-30: -ETC-USD
  2019-03-31: +ATOM-USD
  2019-05-31: +ETC-USD
  2019-06-30: +ALGO-USD, LINK-USD
  2019-07-31: -ETC-USD
  2019-09-30: +HBAR-USD
  2019-12-31: -HBAR-USD
  2020-01-31: +ETC-USD
  2020-02-29: +HBAR-USD
  2020-03-31: -ATOM-USD, ETC-USD
  2020-04-30: +ATOM-USD, ETC-USD
  2020-06-30: -ETC-USD
  2020-07-31: +AVAX-USD, ETC-USD
  2020-08-31: +DOT-USD, SOL-USD
  2020-09-30: +UNI7083-USD
  2020-09-30: -ETC-USD
  2020-10-31: -SOL-USD
  2020-11-30: +AAVE-USD, ETC-USD, NEAR-USD, SOL-USD
  2020-12-31: +GRT6719-USD
  2020-12-31: -ETC-USD, SOL-USD
  2021-01-31: +DOGE-USD, ETC-USD, SOL-USD
  2021-02-28: +INJ-USD, STX4847-USD
  2021-05-31: -INJ-USD
  2021-08-31: +INJ-USD
  2021-09-30: -INJ-USD
  202


--- Index Composition Over Time ---
